# Capítulo 3 · NumPy desde cero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_03/01_numpy.ipynb)

**Fundamentos para Machine Learning**

---

### ¿Por qué empezar por NumPy?

En Machine Learning **todo se reduce a números organizados en arreglos**:

| Dato del mundo real | Cómo lo ve un modelo de ML |
|---|---|
| Un dataset tabular de 1000 clientes con 5 variables | matriz de forma `(1000, 5)` |
| Una imagen en escala de grises de 28×28 píxeles | matriz de forma `(28, 28)` |
| Un lote de 32 fotos a color de 64×64 | tensor de forma `(32, 64, 64, 3)` |
| Los pesos de una capa de una red neuronal | matriz de forma `(n_entradas, n_salidas)` |

**NumPy** (*Numerical Python*) es la librería que permite trabajar con estos arreglos de forma rápida y cómoda. Es la base sobre la que se construyen **pandas**, **scikit-learn** y **matplotlib**, y su forma de trabajar inspiró directamente a **PyTorch** y **TensorFlow**. Si dominas NumPy, cualquier otra herramienta de ML te resultará familiar.

### 🎯 Objetivos

Al terminar este notebook serás capaz de:
1. Crear y manipular arrays de cualquier dimensión.
2. Indexar, filtrar y cambiar la forma de los datos.
3. Escribir código **vectorizado** (sin bucles).
4. Resumir datos con agregaciones y el parámetro `axis`.

### 📚 Índice
1. ¿Qué es un array y por qué no usar listas?
2. Crear arrays
3. Atributos y tipos de datos (`dtype`)
4. Indexación, *slicing* y máscaras booleanas
5. Cambiar la forma de los arrays
6. Operaciones vectorizadas
7. Agregaciones y el parámetro `axis`

In [ ]:
import numpy as np

print("Versión de NumPy:", np.__version__)

> 💡 `np` es el alias universal de NumPy. En cualquier código de ML que leas verás `import numpy as np`.

---
## 1. ¿Qué es un array y por qué no usar listas?

Un **array** (`ndarray`) es una colección de elementos **del mismo tipo** guardados de forma contigua en memoria. Parece una lista de Python, pero se comporta distinto:

In [ ]:
lista = [1, 2, 3, 4]
arr = np.array([1, 2, 3, 4])

print("lista * 2 →", lista * 2)   # repite la lista
print("arr * 2   →", arr * 2)     # multiplica cada elemento

print("lista + lista →", lista + lista)  # concatena
print("arr + arr     →", arr + arr)      # suma elemento a elemento

Con NumPy las operaciones matemáticas se aplican **elemento a elemento**, que es justo lo que necesitamos al trabajar con datos.

Además, NumPy es **muchísimo más rápido** porque las operaciones se ejecutan en código C optimizado, no en un bucle de Python. Comprobémoslo:

In [ ]:
import time

n = 1_000_000
lista = list(range(n))
arr = np.arange(n)

t0 = time.perf_counter()
cuadrados_lista = [x ** 2 for x in lista]
t_lista = time.perf_counter() - t0

t0 = time.perf_counter()
cuadrados_arr = arr ** 2
t_arr = time.perf_counter() - t0

print(f"Lista de Python: {t_lista * 1000:7.1f} ms")
print(f"Array de NumPy:  {t_arr * 1000:7.1f} ms")
print(f"→ NumPy es ~{t_lista / t_arr:.0f} veces más rápido")

Cuando entrenes un modelo con millones de datos, esta diferencia es la que separa minutos de horas. A escribir código sin bucles se le llama **vectorizar**, y es la habilidad más importante de este notebook.

---
## 2. Crear arrays

### 2.1 A partir de listas

In [ ]:
# 1 dimensión (vector)
v = np.array([3, 1, 4, 1, 5])
print(v)

# 2 dimensiones (matriz): una lista de listas
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M)

# 3 dimensiones (tensor): por ejemplo, 2 imágenes de 2x3 píxeles
T = np.array([[[1, 2, 3], [4, 5, 6]],
              [[7, 8, 9], [10, 11, 12]]])
print(T)

### 2.2 Arrays "prefabricados"

Muy habituales para inicializar pesos, crear máscaras o reservar memoria:

In [ ]:
print("zeros:\n", np.zeros((2, 3)))        # todo ceros
print("ones:\n", np.ones((2, 3)))          # todo unos
print("full:\n", np.full((2, 3), 7))       # todo un valor
print("eye (identidad):\n", np.eye(3))     # matriz identidad

In [ ]:
# Secuencias numéricas
print(np.arange(0, 10, 2))       # como range(): inicio, fin (excluido), paso
print(np.linspace(0, 1, 5))      # 5 valores equiespaciados entre 0 y 1 (ambos incluidos)

> 💡 **`arange` vs `linspace`**: usa `arange` cuando conoces el **paso**, y `linspace` cuando conoces **cuántos puntos** quieres. `linspace` es la favorita para graficar funciones (lo veremos al hacer gráficas).

In [ ]:
# Crear arrays con la misma forma que otro
X = np.array([[1.5, 2.0], [3.1, 4.2]])
print(np.zeros_like(X))
print(np.ones_like(X))

---
## 3. Atributos y tipos de datos

Todo array tiene cuatro atributos que vas a consultar **constantemente** mientras depuras código de ML:

| Atributo | Qué indica |
|---|---|
| `.shape` | forma: tamaño de cada dimensión |
| `.ndim` | número de dimensiones |
| `.size` | número total de elementos |
| `.dtype` | tipo de dato de los elementos |

In [ ]:
X = np.zeros((100, 4))   # imagina 100 muestras con 4 características cada una

print("shape:", X.shape)
print("ndim: ", X.ndim)
print("size: ", X.size)
print("dtype:", X.dtype)

### 3.1 Tipos de datos (`dtype`)

In [ ]:
a = np.array([1, 2, 3])
b = np.array([1.0, 2.0, 3.0])
c = np.array([1, 2.5, 3])         # mezcla → NumPy convierte todo a float
d = np.array([True, False, True])

for nombre, x in [("a", a), ("b", b), ("c", c), ("d", d)]:
    print(nombre, x, x.dtype)

In [ ]:
# Convertir de tipo con astype
x = np.array([1.7, 2.2, 3.9])
print(x.astype(int))          # trunca los decimales
print(x.astype(np.float32))   # menos precisión, la mitad de memoria

> 🧠 **En ML**: las redes neuronales suelen trabajar en `float32` (o incluso `float16`) porque ocupa la mitad de memoria que `float64` y las GPUs son más rápidas con él. Comprobémoslo:

In [ ]:
grande64 = np.zeros((1000, 1000), dtype=np.float64)
grande32 = np.zeros((1000, 1000), dtype=np.float32)
print(f"float64: {grande64.nbytes / 1e6:.0f} MB")
print(f"float32: {grande32.nbytes / 1e6:.0f} MB")

---
## 4. Indexación, *slicing* y máscaras booleanas

### 4.1 Arrays de 1 dimensión
Funciona igual que con listas de Python: los índices empiezan en `0` y los negativos cuentan desde el final.

In [ ]:
a = np.array([10, 20, 30, 40, 50, 60])

print(a[0])      # primer elemento
print(a[-1])     # último elemento
print(a[1:4])    # del índice 1 al 3 (el 4 no se incluye)
print(a[:3])     # los 3 primeros
print(a[::2])    # uno sí, uno no
print(a[::-1])   # invertido

### 4.2 Arrays de 2 dimensiones
La sintaxis es `M[filas, columnas]`. En ML, por convención:
- **filas = muestras** (cada ejemplo, cliente, imagen…)
- **columnas = características** o *features* (edad, precio, píxel…)

In [ ]:
M = np.array([[ 1,  2,  3,  4],
              [ 5,  6,  7,  8],
              [ 9, 10, 11, 12]])

print("Elemento fila 1, columna 2:", M[1, 2])
print("Fila 0 completa:          ", M[0])        # equivale a M[0, :]
print("Columna 1 completa:       ", M[:, 1])
print("Submatriz:\n", M[0:2, 1:3])

### 4.3 Máscaras booleanas: filtrar datos ⭐

Esta es una de las herramientas más usadas en análisis de datos. Una comparación sobre un array produce un **array de booleanos**, que podemos usar para filtrar:

In [ ]:
edades = np.array([15, 22, 37, 8, 45, 19, 63])

mascara = edades >= 18
print("Máscara:", mascara)
print("Mayores de edad:", edades[mascara])

# Directamente en una línea
print("Menores:", edades[edades < 18])

In [ ]:
# np.where: "si se cumple, esto; si no, aquello"
categoria = np.where(edades >= 18, "adulto", "menor")
print(categoria)

In [ ]:
X = np.array([[1.0, 2.0],
              [1.5, 1.8],
              [5.0, 8.0],
              [6.0, 9.0]])
y = np.array([0, 0, 1, 1])

print("Muestras de la clase 1:\n", X[y == 1])

---
## 5. Cambiar la forma de los arrays

### 5.1 `reshape`
Reorganiza los mismos datos en otra forma. El número total de elementos debe coincidir.

In [ ]:
a = np.arange(12)
print(a.shape)

print(a.reshape(3, 4))
print(a.reshape(2, 6))
print(a.reshape(4, -1))   # -1 = "calcula tú esta dimensión"

> 🧠 **En ML**: el `-1` es muy útil. Por ejemplo, para "aplanar" imágenes antes de pasarlas a un modelo clásico o a una capa densa:

In [ ]:
# 60 imágenes de 28x28 píxeles (como el famoso dataset MNIST)
imagenes = np.zeros((60, 28, 28))
print("Antes:  ", imagenes.shape)

planas = imagenes.reshape(60, -1)     # cada imagen pasa a ser un vector de 784
print("Después:", planas.shape)

In [ ]:
# scikit-learn exige que X sea 2D: (n_muestras, n_features).
# Si tienes una sola feature como vector 1D, conviértela en columna:
x = np.array([1, 2, 3, 4])
print(x.shape)
print(x.reshape(-1, 1).shape)     # opción 1
print(x[:, np.newaxis].shape)     # opción 2 (equivalente)

### 5.2 Transponer y aplanar

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print("Original", M.shape, "\n", M)
print("Transpuesta", M.T.shape, "\n", M.T)
print("Aplanada:", M.ravel())

### 5.3 Unir arrays

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("Una debajo de otra (axis=0):\n", np.concatenate([a, b], axis=0))
print("Una al lado de otra (axis=1):\n", np.concatenate([a, b], axis=1))

# Atajos equivalentes
print(np.vstack([a, b]).shape, np.hstack([a, b]).shape)

In [ ]:
# np.stack crea una NUEVA dimensión: útil para formar un "lote" (batch)
img1 = np.zeros((28, 28))
img2 = np.ones((28, 28))
lote = np.stack([img1, img2])
print(lote.shape)    # (2, 28, 28)

In [ ]:
# Ejemplo ML: añadir una columna de unos a X (el "sesgo" o bias en regresión lineal)
X = np.array([[2.0, 3.0],
              [4.0, 5.0],
              [6.0, 7.0]])
X_con_bias = np.hstack([np.ones((X.shape[0], 1)), X])
print(X_con_bias)

---
## 6. Operaciones vectorizadas

Las operaciones aritméticas se aplican elemento a elemento, sin bucles:

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("a + b =", a + b)
print("a * b =", a * b)
print("b / a =", b / a)
print("a ** 2 =", a ** 2)
print("a > 2 =", a > 2)

### Funciones universales (*ufuncs*)
NumPy incluye versiones vectorizadas de todas las funciones matemáticas:

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])

print("exp:  ", np.exp(x))
print("log1p:", np.log1p(x))      # log(1 + x), evita log(0)
print("sqrt: ", np.sqrt(x))
print("sin:  ", np.sin(x))
print("abs:  ", np.abs(np.array([-3, 2, -1])))
print("clip: ", np.clip(np.array([-5, 0.5, 10]), 0, 1))  # recorta al rango [0, 1]

In [ ]:
notas = np.array([[7.0, 8.5, 6.0],
                  [9.0, 9.5, 8.0],
                  [5.0, 6.0, 4.5],
                  [8.0, 7.0, 9.0]])

print("Media global:", notas.mean())